# Home Credit Default Risk
## Sprint 3 — Modelagem, Comparação de Modelos e Ajuste de Hiperparâmetros

**Período:** 05/06 a 14/06

Este notebook é a continuação das Sprints 1 (Análise Exploratória) e 2 (Pré-processamento e Feature Engineering). Aqui construímos, comparamos e ajustamos modelos de classificação para prever a inadimplência (`TARGET`).

**Regras importantes desta sprint:**
- O conjunto de teste (`X_test_final`, `y_test`) só pode ser usado **uma única vez**, na Seção 6.
- Toda comparação e ajuste de modelos usa apenas o conjunto de treino com cross-validation (`cv=5`).
- Problema: classificação **desbalanceada** (8% classe 1) → métrica principal = **AUC-ROC**; complementar = F1-Macro.

**Estrutura do notebook:**

| Seção | Conteúdo |
|---|---|
| 1 | Reprodução do Pipeline da Sprint 2 → dados prontos |
| 2 | Baseline (DummyClassifier) — piso mínimo de desempenho |
| 3 | Comparação de 4 modelos via cross-validation |
| 4 | Ajuste de hiperparâmetros para os 2 melhores modelos |
| 5 | Escolha justificada do modelo final |
| 6 | Avaliação final no conjunto de teste (uma única vez) |
| 7 | Persistência do modelo com joblib |

---
## SEÇÃO 1 — Carregamento do Pipeline da Sprint 2

Reproduzimos o pipeline de pré-processamento construído na Sprint 2 de forma compacta e sequencial. Todos os `fit()` são executados **exclusivamente no conjunto de treino** para garantir ausência de *data leakage*.

| Etapa | Técnica | Justificativa |
|---|---|---|
| Split 80/20 | `train_test_split(stratify=y)` | Preserva proporção 8% classe 1 em ambos os splits |
| Sentinela | `DAYS_EMPLOYED=365243 → NaN` (antes da imputação) | Código de aposentados inflaria a mediana |
| Missing | Drop >60% nulos + mediana (num.) + moda (cat.) | Mediana robusta a outliers; moda para categóricas |
| Outliers | Winsorização IQR (clip ≤ 10% outliers) | Preserva shape real, apenas capa caudas extremas |
| Encoding | LabelEncoder (ordinais) + OHE (≤10 cat.) + Target Encoding (alta cardinalidade) | Estratégia adequada a cada tipo semântico |
| Feature Eng. | 4 ratios financeiros/temporais | Alavancagem e estabilidade de emprego como proxy de risco |
| Scaling | `StandardScaler` + `RobustScaler` (se >5% outliers) | Modelos lineares e baseados em distância precisam de escala |
| Seleção | `VarianceThreshold` + correlação + Top-60 RF importance | Remove ruído sem destruir sinal útil |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import (
    train_test_split, cross_validate,
    GridSearchCV, RandomizedSearchCV
)
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import VarianceThreshold
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score, f1_score, accuracy_score, precision_score, recall_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report, RocCurveDisplay
)
import joblib
import os

RANDOM_STATE  = 42
CV_FOLDS      = 5
SCORING       = 'roc_auc'
SAMPLE_SIZE   = 50_000

sns.set_theme(style='whitegrid')

# ─── 1.1 Carregamento e Split (idêntico à Sprint 2) ──────────────────────────
df_raw = pd.read_csv('../data/raw/application_train.csv')
X = df_raw.drop(columns=['TARGET', 'SK_ID_CURR'])
y = df_raw['TARGET']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f"Dataset: {df_raw.shape} | X_train: {X_train.shape} | X_test: {X_test.shape}")
print(f"Distribuição TARGET — 0: {(y==0).mean():.2%} | 1: {(y==1).mean():.2%}")

# ─── 1.2 Sentinela + Missing Values ──────────────────────────────────────────
# Fix BEFORE imputation: sentinela 365243 → NaN (aposentados sem vínculo)
# Convertido antes da imputação para que a mediana seja calculada sem o sentinela
X_train['DAYS_EMPLOYED'] = X_train['DAYS_EMPLOYED'].replace(365243, np.nan)
X_test['DAYS_EMPLOYED']  = X_test['DAYS_EMPLOYED'].replace(365243, np.nan)

# Drop colunas com >60% de nulos (ruído excessivo)
miss_pct  = X_train.isnull().mean()
cols_drop = miss_pct[miss_pct > 0.60].index.tolist()
X_train = X_train.drop(columns=cols_drop)
X_test  = X_test.drop(columns=cols_drop)
print(f"\n[Missing] {len(cols_drop)} colunas removidas (>60% nulos) → shape: {X_train.shape}")

# Imputação mediana — numéricas (mediana robusta a outliers)
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
med_imp  = SimpleImputer(strategy='median')
X_train[num_cols] = med_imp.fit_transform(X_train[num_cols])
X_test[num_cols]  = med_imp.transform(X_test[num_cols])

# Imputação moda — categóricas
cat_cols = X_train.select_dtypes(include='object').columns.tolist()
for col in cat_cols:
    moda = X_train[col].mode()[0]
    X_train[col] = X_train[col].fillna(moda)
    X_test[col]  = X_test[col].fillna(moda)
print(f"[Missing] Imputação concluída — nulos restantes: {X_train.isnull().sum().sum()}")

# ─── 1.3 Outliers — Winsorização IQR ────────────────────────────────────────
# HOUR_APPR_PROCESS_START: horários fora do comercial são válidos (não outliers)
skip_win = {'HOUR_APPR_PROCESS_START'}
for col in X_train.select_dtypes(include=[np.number]).columns:
    if col in skip_win:
        continue
    Q1, Q3 = X_train[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    if IQR == 0:
        continue
    out_pct = ((X_train[col] < Q1 - 1.5*IQR) | (X_train[col] > Q3 + 1.5*IQR)).mean()
    if out_pct > 0.10:
        continue  # assimetria estrutural: não é outlier isolado
    lo, hi = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    X_train[col] = X_train[col].clip(lo, hi)
    X_test[col]  = X_test[col].clip(lo, hi)
print("[Outliers] Winsorização IQR aplicada")

# ─── 1.4 Encoding ────────────────────────────────────────────────────────────
# LabelEncoder (ordinais — preserva hierarquia natural)
ordinal_maps = {
    'NAME_EDUCATION_TYPE': ['Lower secondary', 'Secondary / secondary special',
                            'Incomplete higher', 'Higher education', 'Academic degree'],
    'HOUSETYPE_MODE':      ['terraced house', 'specific housing', 'block of flats'],
}
for col, order in ordinal_maps.items():
    if col not in X_train.columns:
        continue
    mapping = {cat: idx for idx, cat in enumerate(order)}
    X_train[col] = X_train[col].map(mapping)
    X_test[col]  = X_test[col].map(mapping)

# OneHotEncoder (nominais ≤10 categorias — sem ordem entre categorias)
ohe_cols = ['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY',
            'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_FAMILY_STATUS',
            'NAME_HOUSING_TYPE', 'WEEKDAY_APPR_PROCESS_START',
            'WALLSMATERIAL_MODE', 'EMERGENCYSTATE_MODE']
ohe_cols = [c for c in ohe_cols if c in X_train.columns]
X_train = pd.get_dummies(X_train, columns=ohe_cols, drop_first=True, dtype=int)
X_test  = pd.get_dummies(X_test,  columns=ohe_cols, drop_first=True, dtype=int)
X_test  = X_test.reindex(columns=X_train.columns, fill_value=0)

# Target Encoding (alta cardinalidade — taxa média de inadimplência por categoria)
te_cols = [c for c in ['ORGANIZATION_TYPE', 'OCCUPATION_TYPE'] if c in X_train.columns]
for col in te_cols:
    te_map = y_train.groupby(X_train[col]).mean()
    X_train[col] = X_train[col].map(te_map)
    X_test[col]  = X_test[col].map(te_map).fillna(y_train.mean())
print(f"[Encoding] Concluído → shape: {X_train.shape}")

# ─── 1.5 Feature Engineering ─────────────────────────────────────────────────
# Grupo A: Ratios de alavancagem (risco financeiro)
X_train['CREDIT_TO_INCOME']  = X_train['AMT_CREDIT']  / (X_train['AMT_INCOME_TOTAL'] + 1)
X_test['CREDIT_TO_INCOME']   = X_test['AMT_CREDIT']   / (X_test['AMT_INCOME_TOTAL']  + 1)

X_train['ANNUITY_TO_INCOME'] = X_train['AMT_ANNUITY'] / (X_train['AMT_INCOME_TOTAL'] + 1)
X_test['ANNUITY_TO_INCOME']  = X_test['AMT_ANNUITY']  / (X_test['AMT_INCOME_TOTAL']  + 1)

X_train['CREDIT_TO_ANNUITY'] = X_train['AMT_CREDIT']  / (X_train['AMT_ANNUITY'] + 1)
X_test['CREDIT_TO_ANNUITY']  = X_test['AMT_CREDIT']   / (X_test['AMT_ANNUITY']  + 1)

# Grupo B: Indicador de estabilidade de emprego
X_train['EMPLOYMENT_DAYS'] = -X_train['DAYS_EMPLOYED'] / 365.25
X_test['EMPLOYMENT_DAYS']  = -X_test['DAYS_EMPLOYED']  / 365.25
print(f"[FE] 4 features criadas → shape: {X_train.shape}")

# ─── 1.6 Scaling ─────────────────────────────────────────────────────────────
# RobustScaler para features com >5% outliers residuais; StandardScaler para as demais
num_final = X_train.select_dtypes(include=[np.number]).columns.tolist()
feat_robust, feat_std = [], []
for col in num_final:
    Q1, Q3 = X_train[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    if IQR == 0:
        feat_std.append(col)
        continue
    out_pct = ((X_train[col] < Q1 - 1.5*IQR) | (X_train[col] > Q3 + 1.5*IQR)).mean()
    (feat_robust if out_pct > 0.05 else feat_std).append(col)

if feat_std:
    sc_std = StandardScaler()
    X_train[feat_std] = sc_std.fit_transform(X_train[feat_std])
    X_test[feat_std]  = sc_std.transform(X_test[feat_std])
if feat_robust:
    sc_rob = RobustScaler()
    X_train[feat_robust] = sc_rob.fit_transform(X_train[feat_robust])
    X_test[feat_robust]  = sc_rob.transform(X_test[feat_robust])
print(f"[Scaling] StandardScaler: {len(feat_std)} | RobustScaler: {len(feat_robust)} → shape: {X_train.shape}")

In [ ]:
# ─── 1.7 Seleção de Features (Sprint 2 — Seção 7) ────────────────────────────

# Etapa 1: VarianceThreshold — remove quasi-constantes (var < 0.01)
sel_var      = VarianceThreshold(threshold=0.01)
sel_var.fit(X_train)
low_var_cols = X_train.columns[~sel_var.get_support()].tolist()

# Etapa 2: Filtro de correlação com TARGET (|corr| < 0.005 → sinal mínimo indetectável)
corr_abs      = X_train.corrwith(y_train).abs()
low_corr_cols = corr_abs[corr_abs < 0.005].index.tolist()

remove_init = list(set(low_var_cols) | set(low_corr_cols))
X_pf      = X_train.drop(columns=remove_init, errors='ignore')
X_pf_test = X_test.drop(columns=remove_init,  errors='ignore')
print(f"[Seleção] Removidas {len(remove_init)} features (var/corr) → {X_pf.shape[1]} restantes")

# Etapa 3: Feature Importance via RandomForest (top 60)
# n_estimators=30 e max_depth=5 para velocidade (seleção, não modelo final)
rf_sel = RandomForestClassifier(
    n_estimators=30, max_depth=5, n_jobs=-1,
    random_state=RANDOM_STATE, class_weight='balanced'
)
rf_sel.fit(X_pf, y_train)

imp_df = pd.DataFrame({'feature': X_pf.columns, 'importance': rf_sel.feature_importances_})
imp_df = imp_df.sort_values('importance', ascending=False).reset_index(drop=True)

# Garantir que as features de FE estejam no conjunto final (interpretação de negócio clara)
fe_feats = ['CREDIT_TO_INCOME', 'ANNUITY_TO_INCOME', 'CREDIT_TO_ANNUITY', 'EMPLOYMENT_DAYS']
top60    = imp_df.head(60)['feature'].tolist()
selected = list(dict.fromkeys(top60 + [f for f in fe_feats if f in X_pf.columns and f not in top60]))
selected = [f for f in selected if f in X_pf.columns]

X_train_final = X_pf[selected].copy()
X_test_final  = X_pf_test[selected].copy()
print(f"[Seleção] Top features RF → shape final: {X_train_final.shape}")

# ─── Confirmação dos Shapes ───────────────────────────────────────────────────
print("\n" + "="*55)
print("CONFIRMAÇÃO — DADOS PRONTOS PARA MODELAGEM")
print("="*55)
print(f"X_train_final : {X_train_final.shape}")
print(f"X_test_final  : {X_test_final.shape}")
print(f"y_train       : {y_train.shape}   (TARGET=1: {y_train.mean():.2%})")
print(f"y_test        : {y_test.shape}    (TARGET=1: {y_test.mean():.2%})")
assert X_train_final.isnull().sum().sum() == 0, "ERRO: nulos no treino!"
assert X_test_final.isnull().sum().sum()  == 0, "ERRO: nulos no teste!"
print("\nSem nulos. Ambiente pronto para modelagem.")

---
## SEÇÃO 2 — Baseline

O baseline define o **piso mínimo** de desempenho: qualquer modelo treinado deve superá-lo para justificar sua existência. Utilizamos o `DummyClassifier(strategy='most_frequent')`, que sempre prediz a classe majoritária (0 = adimplente).

**Subamostra para Cross-Validation:** os 246k registros de treino tornam cada fold do CV muito lento para alguns modelos. Utilizamos uma **subamostra estratificada de 50k** (mesma proporção de classes) para todas as avaliações de CV desta sprint — mesma abordagem da Sprint 2. O modelo final (Seção 6) é treinado no conjunto completo.

**Métricas avaliadas:**
- `AUC-ROC` (principal): robusta ao desbalanceamento, avalia o poder de separação
- `F1-Macro` (complementar): penaliza igualmente erros na classe minoritária

In [ ]:
# Subamostra estratificada de 50k para todas as avaliações de CV desta sprint
X_cv, _, y_cv, _ = train_test_split(
    X_train_final, y_train,
    train_size=SAMPLE_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_train,
)
print(f"Subamostra CV: {X_cv.shape} | TARGET=1: {y_cv.mean():.2%}")

# Baseline: DummyClassifier sempre prediz classe 0 (majoritária)
baseline = DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE)

scoring_multi = {'auc': 'roc_auc', 'f1_macro': 'f1_macro'}
bl_cv = cross_validate(baseline, X_cv, y_cv, cv=CV_FOLDS, scoring=scoring_multi)

baseline_auc_mean = bl_cv['test_auc'].mean()
baseline_auc_std  = bl_cv['test_auc'].std()
baseline_f1_mean  = bl_cv['test_f1_macro'].mean()
baseline_f1_std   = bl_cv['test_f1_macro'].std()

print("\n" + "="*60)
print("BASELINE — DummyClassifier (strategy='most_frequent')")
print("="*60)
print(f"AUC-ROC  : {baseline_auc_mean:.4f} ± {baseline_auc_std:.4f}")
print(f"F1-Macro : {baseline_f1_mean:.4f} ± {baseline_f1_std:.4f}")
print()
print("Interpretação: ao prever sempre classe 0, a acurácia é 91.9%,")
print("mas nenhum inadimplente é detectado (Recall classe 1 = 0).")
print("AUC = 0.50 equivale a um classificador aleatório — piso mínimo.")

In [ ]:
# Visualização: desbalanceamento + referência de AUC
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Gráfico 1: distribuição das classes
dist = y_train.value_counts(normalize=True).sort_index()
bars = axes[0].bar(['0 (Adimplente)', '1 (Inadimplente)'],
                   [dist[0]*100, dist[1]*100],
                   color=['steelblue', 'tomato'], edgecolor='black', alpha=0.85)
axes[0].set_ylabel('Proporção (%)')
axes[0].set_title('Distribuição do TARGET — Conjunto de Treino')
for bar, val in zip(bars, [dist[0]*100, dist[1]*100]):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 0.3,
                 f'{val:.1f}%', ha='center', fontweight='bold')

# Gráfico 2: baseline AUC vs mínimo esperado
labels = ['Baseline\n(DummyClassifier)', 'Meta mínima\nesperada']
values = [baseline_auc_mean, 0.70]
colors = ['gray', 'steelblue']
axes[1].barh(labels, values, color=colors, alpha=0.85, edgecolor='black')
axes[1].axvline(x=0.5, color='red', linestyle='--', linewidth=1.2, label='Aleatório (AUC=0.50)')
axes[1].set_xlim(0.40, 0.80)
axes[1].set_xlabel('AUC-ROC')
axes[1].set_title('Referência: Baseline vs Meta Mínima')
axes[1].legend()
for i, v in enumerate(values):
    axes[1].text(v + 0.003, i, f'{v:.2f}', va='center', fontweight='bold')

plt.suptitle('Sprint 3 — Contexto do Problema e Referências', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Por que a acurácia não é a métrica principal

Com 91,9% da classe 0, um modelo que sempre prediz "adimplente" teria **91,9% de acurácia** — número impressionante que não reflete nenhuma capacidade preditiva real. O modelo estaria falhando em 100% dos casos que realmente importam (inadimplentes).

**Custo assimétrico dos erros no crédito:**

| Erro | Descrição | Custo para o banco |
|---|---|---|
| **Falso Negativo (FN)** | Approvar crédito para inadimplente | Perda financeira **direta** do capital |
| **Falso Positivo (FP)** | Negar crédito a bom pagador | Custo de oportunidade (juros não recebidos) |

O FN é o erro mais grave. Por isso, maximizar a **detecção de inadimplentes** (Recall classe 1) é a prioridade estratégica. A AUC-ROC captura esse poder discriminativo sem depender do threshold, sendo ideal para problemas desbalanceados onde queremos otimizar a curva de risco-retorno.

**Meta desta sprint:** superar AUC-ROC > 0.70 com desvio padrão < 0.015 entre os folds do CV.

---
## SEÇÃO 3 — Treinamento e Comparação de Modelos

Testamos **4 algoritmos** que representam diferentes famílias de classificadores. Todos são avaliados sob as mesmas condições (mesma subamostra de 50k, `cv=5`, `scoring='roc_auc'`), garantindo comparação justa.

**Critério de comparação:**
1. **AUC-ROC média** — métrica principal (maior é melhor)
2. **Desvio padrão da AUC** — estabilidade entre folds (menor é melhor)

> Um modelo com AUC média ligeiramente inferior mas **muito mais estável** pode ser preferível em produção, pois garante comportamento previsível em dados novos.

Os 2 modelos com melhor combinação de AUC e estabilidade serão selecionados para ajuste de hiperparâmetros na Seção 4.

### 3.1 Justificativa dos Algoritmos Escolhidos

Os 4 algoritmos foram escolhidos para representar **famílias distintas** de aprendizado, permitindo uma comparação abrangente do espectro disponível:

| Modelo | Família | `class_weight` | Justificativa da escolha |
|---|---|---|---|
| `LogisticRegression` | Linear (generativo) | `'balanced'` | Referência interpretável; coeficientes diretos permitem análise regulatória (Basel III/IRB) |
| `DecisionTree` | Árvore única | `'balanced'` | Não-linear mas sem ensemble; útil para evidenciar a instabilidade de modelos simples (alta variância) |
| `RandomForest` | Bagging (ensemble) | `'balanced'` | Reduz variância por média de árvores; robusto a outliers e ruído nas features |
| `HistGradientBoosting` | Boosting (ensemble) | `'balanced'` | Estado da arte para dados tabulares; treina iterativamente para corrigir erros dos modelos anteriores |

**Por que `class_weight='balanced'`:**
Com 91,9% vs 8,1% de classes, modelos sem ponderação tendem a ignorar a classe minoritária. O parâmetro ajusta automaticamente o peso de cada amostra inversamente à frequência de sua classe:

`w_classe = n_amostras / (n_classes × n_amostras_da_classe)`

Assim, cada inadimplente (classe 1) recebe ~11× mais peso que um adimplente (classe 0), forçando o modelo a priorizar a detecção de inadimplentes.

**Nota sobre os Pipelines:** Os dados já chegam pré-processados da Sprint 2 (`X_train_final`). Os Pipelines abaixo encapsulam apenas o classificador, garantindo a API consistente de `fit`/`predict`/`predict_proba` requerida pelo `cross_validate`.

In [ ]:
# Definição dos 4 modelos em Pipeline
# Os dados já estão pré-processados (X_train_final) — Pipeline encapsula apenas o classificador
modelos = {
    'LogisticRegression': Pipeline([
        ('model', LogisticRegression(
            class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE
        ))
    ]),
    'DecisionTree': Pipeline([
        ('model', DecisionTreeClassifier(
            class_weight='balanced', max_depth=10, random_state=RANDOM_STATE
        ))
    ]),
    'RandomForest': Pipeline([
        ('model', RandomForestClassifier(
            n_estimators=100, class_weight='balanced',
            n_jobs=-1, random_state=RANDOM_STATE
        ))
    ]),
    'HistGradientBoosting': Pipeline([
        ('model', HistGradientBoostingClassifier(
            class_weight='balanced', random_state=RANDOM_STATE
        ))
    ]),
}

print("Modelos definidos:")
for nome, pipe in modelos.items():
    clf = pipe.named_steps['model']
    print(f"  {nome}: {clf.__class__.__name__}({', '.join(f'{k}={v}' for k,v in clf.get_params().items() if v is not None and k in ['class_weight','max_iter','n_estimators','max_depth'])})")

In [ ]:
# Cross-validation: AUC-ROC e F1-Macro em subamostra de 50k (cv=5)
scoring_multi = {'auc': 'roc_auc', 'f1_macro': 'f1_macro'}
cv_results = {}

print(f"Cross-validation (cv={CV_FOLDS}, n=50k) — aguarde...")
print("-" * 70)

for nome, pipeline in modelos.items():
    print(f"Avaliando {nome}...", end=' ', flush=True)
    res = cross_validate(pipeline, X_cv, y_cv, cv=CV_FOLDS,
                         scoring=scoring_multi, n_jobs=1)
    cv_results[nome] = {
        'auc_scores': res['test_auc'],
        'auc_mean':   res['test_auc'].mean(),
        'auc_std':    res['test_auc'].std(),
        'f1_mean':    res['test_f1_macro'].mean(),
        'f1_std':     res['test_f1_macro'].std(),
    }
    print(f"AUC = {res['test_auc'].mean():.4f} ± {res['test_auc'].std():.4f} | "
          f"F1-macro = {res['test_f1_macro'].mean():.4f} ± {res['test_f1_macro'].std():.4f}")

print("\nCross-validation concluído.")

In [ ]:
# Tabela comparativa — ordenada por AUC Médio
obs = {
    'LogisticRegression':   'Linear, interpretável — estável entre folds',
    'DecisionTree':         'Não-linear, alta variância — instável (único)',
    'RandomForest':         'Bagging — reduz variância por média de árvores',
    'HistGradientBoosting': 'Boosting — estado da arte para dados tabulares',
}

tabela = pd.DataFrame([
    {
        'Modelo':       nome,
        'AUC Médio':    round(v['auc_mean'], 4),
        'Desvio AUC':   round(v['auc_std'],  4),
        'F1-Macro':     round(v['f1_mean'],  4),
        'Desvio F1':    round(v['f1_std'],   4),
        'Observação':   obs.get(nome, ''),
    }
    for nome, v in cv_results.items()
]).sort_values('AUC Médio', ascending=False).reset_index(drop=True)

display(tabela)
print(f"\nBaseline (DummyClassifier): AUC = {baseline_auc_mean:.4f} | F1-Macro = {baseline_f1_mean:.4f}")
superou = all(v['auc_mean'] > baseline_auc_mean for v in cv_results.values())
print(f"Todos superam o baseline?  {'Sim ' if superou else 'Não '}")

In [ ]:
# Gráfico comparativo com barras de erro (desvio padrão entre folds)
nomes     = list(cv_results.keys())
auc_means = [cv_results[n]['auc_mean'] for n in nomes]
auc_stds  = [cv_results[n]['auc_std']  for n in nomes]
f1_means  = [cv_results[n]['f1_mean']  for n in nomes]
f1_stds   = [cv_results[n]['f1_std']   for n in nomes]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# AUC-ROC
bars1 = axes[0].bar(nomes, auc_means, yerr=auc_stds, capsize=6,
                    color='steelblue', alpha=0.85, edgecolor='black')
axes[0].axhline(y=baseline_auc_mean, color='red', linestyle='--',
                linewidth=1.2, label=f'Baseline ({baseline_auc_mean:.4f})')
axes[0].set_title('AUC-ROC por Modelo (CV=5, n=50k)', fontsize=11, fontweight='bold')
axes[0].set_ylabel('AUC-ROC')
axes[0].set_ylim(max(0.40, min(auc_means) - 0.05), min(1.0, max(auc_means) + 0.05))
axes[0].tick_params(axis='x', rotation=15)
axes[0].legend()
for bar, val, std in zip(bars1, auc_means, auc_stds):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + std + 0.003,
                 f'{val:.4f}', ha='center', fontsize=9, fontweight='bold')

# F1-Macro
bars2 = axes[1].bar(nomes, f1_means, yerr=f1_stds, capsize=6,
                    color='coral', alpha=0.85, edgecolor='black')
axes[1].axhline(y=baseline_f1_mean, color='red', linestyle='--',
                linewidth=1.2, label=f'Baseline ({baseline_f1_mean:.4f})')
axes[1].set_title('F1-Macro por Modelo (CV=5, n=50k)', fontsize=11, fontweight='bold')
axes[1].set_ylabel('F1-Macro')
axes[1].set_ylim(max(0.0, min(f1_means) - 0.05), min(1.0, max(f1_means) + 0.05))
axes[1].tick_params(axis='x', rotation=15)
axes[1].legend()
for bar, val, std in zip(bars2, f1_means, f1_stds):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + std + 0.003,
                 f'{val:.4f}', ha='center', fontsize=9, fontweight='bold')

plt.suptitle('Comparação de Modelos — Cross-Validation\n'
             'Barras de erro = desvio padrão entre folds (estabilidade)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# Ordenar por AUC para facilitar análise
sorted_models = sorted(cv_results.items(), key=lambda x: x[1]['auc_mean'], reverse=True)
print("Ranking por AUC-ROC:")
for i, (nome, v) in enumerate(sorted_models):
    print(f"  {i+1}. {nome}: {v['auc_mean']:.4f} ± {v['auc_std']:.4f}")

### 3.2 Análise dos Resultados — Média vs. Desvio Padrão

**Por que o desvio padrão importa tanto quanto a média:**

Dois modelos com médias próximas não são equivalentes se seus desvios diferirem significativamente:

- Modelo A: AUC = 0.738 ± 0.003 → previsível, opera na faixa [0.732, 0.744] em novos dados
- Modelo B: AUC = 0.742 ± 0.018 → em alguns folds pode cair a 0.706, em outros subir a 0.778

O Modelo A, apesar da média ligeiramente menor, é **muito mais seguro em produção**. Em risco de crédito, instabilidade no desempenho implica em meses em que a carteira é sub ou superprotegida — risco operacional real.

**Análise esperada por modelo:**

| Modelo | Esperado | Por quê |
|---|---|---|
| `DecisionTree` | Alta variância | Árvore única sem ensemble — cada fold pode aprender partições muito diferentes |
| `LogisticRegression` | Baixa variância | Superfície de decisão linear — menos sensível ao conjunto de treino específico |
| `RandomForest` | Variância moderada-baixa | Média de 100 árvores cancela ruído individual |
| `HistGradientBoosting` | Variância baixa, AUC alta | Regularização implícita no boosting + class_weight |

**Modelos selecionados para ajuste:** `LogisticRegression` e `HistGradientBoosting` — combinam as melhores AUC com maior estabilidade. O DecisionTree (alta variância) e RandomForest (geralmente AUC abaixo do boosting neste dataset) ficam de fora.

---
## SEÇÃO 4 — Ajuste de Hiperparâmetros

Refinamos os **2 modelos selecionados** (LogisticRegression e HistGradientBoosting) usando estratégias de busca diferentes, justificadas pelo tamanho do espaço de hiperparâmetros de cada modelo.

**Regras mantidas:**
- Mesma subamostra de 50k e `cv=5` da Seção 3 (comparação justa)
- Mesma métrica: `roc_auc`
- Conjunto de teste **não é tocado**

| Modelo | Estratégia | Justificativa |
|---|---|---|
| `LogisticRegression` | `GridSearchCV` | Poucos hiperparâmetros → busca exaustiva é factível e garante o ótimo do grid |
| `HistGradientBoosting` | `RandomizedSearchCV` | Espaço combinatório grande → busca aleatória explora mais eficientemente |

### 4.1 Modelo 1 — LogisticRegression com GridSearchCV

**Hiperparâmetros e justificativas:**

| Parâmetro | Papel | Grid testado |
|---|---|---|
| `C` | Inverso da regularização L2. C pequeno → mais regularização (mais bias, menos variância). C grande → menos regularização (pode overfit). | [0.01, 0.1, 1, 10, 100] |
| `solver` | Algoritmo de otimização do gradiente. `lbfgs` é rápido para datasets menores; `saga` suporta L1/L2 e é mais eficiente em larga escala. | ['lbfgs', 'saga'] |

**GridSearchCV:** 5 valores de C × 2 solvers = **10 combinações** × `cv=5` = 50 fits. Busca exaustiva é factível (~1-2 min) e garante que encontramos o ótimo do espaço definido.

In [ ]:
param_grid_lr = {
    'model__C':      [0.01, 0.1, 1, 10, 100],
    'model__solver': ['lbfgs', 'saga'],
}

print("GridSearchCV — LogisticRegression")
print(f"Espaço: {param_grid_lr}")
print(f"Combinações: 5 × 2 = 10 × cv={CV_FOLDS} = 50 fits")
print("Executando...\n")

search_lr = GridSearchCV(
    modelos['LogisticRegression'],
    param_grid_lr,
    cv=CV_FOLDS,
    scoring=SCORING,
    n_jobs=-1,
    verbose=0,
    refit=True,
)
search_lr.fit(X_cv, y_cv)

lr_default_auc = cv_results['LogisticRegression']['auc_mean']
lr_tuned_auc   = search_lr.best_score_

print(f"Melhores hiperparâmetros : {search_lr.best_params_}")
print(f"AUC padrão  (cv=5)       : {lr_default_auc:.4f} ± {cv_results['LogisticRegression']['auc_std']:.4f}")
print(f"AUC ajustado (cv=5)      : {lr_tuned_auc:.4f}")
delta_lr = lr_tuned_auc - lr_default_auc
print(f"Variação                 : {delta_lr:+.4f} ({'melhora' if delta_lr > 0.001 else 'marginal' if delta_lr > 0 else 'sem ganho'})")

### 4.2 Modelo 2 — HistGradientBoostingClassifier com RandomizedSearchCV

**Por que RandomizedSearchCV aqui:**
O espaço combinatório total do HistGBT seria enorme no GridSearch (ex: 5×4×4×4×4×4 = 5120 combinações). Com `n_iter=15`, amostramos 15 combinações aleatórias — muito mais eficiente e, na prática, quase tão eficaz para encontrar boas regiões do espaço.

**Hiperparâmetros e justificativas:**

| Parâmetro | Papel | Valores testados |
|---|---|---|
| `learning_rate` | Tamanho do passo de cada árvore. Menor = mais conservador, precisa de mais árvores (`max_iter`) | [0.01, 0.05, 0.1, 0.15, 0.2] |
| `max_iter` | Número de rodadas de boosting (árvores). Mais rodadas com learning_rate baixo | [100, 150, 200, 300] |
| `max_leaf_nodes` | Controla a complexidade de cada árvore. Árvores mais rasas = menos overfitting | [15, 31, 63, 127] |
| `min_samples_leaf` | Regularização implícita: folhas precisam de N amostras mínimas | [10, 20, 50, 100] |
| `l2_regularization` | Penalidade L2 nos pesos das folhas | [0.0, 0.1, 0.5, 1.0] |
| `max_depth` | Profundidade máxima por árvore (`None` = sem limite, controlado por `max_leaf_nodes`) | [3, 5, 7, None] |

In [ ]:
param_dist_hgbt = {
    'model__learning_rate':    [0.01, 0.05, 0.1, 0.15, 0.2],
    'model__max_iter':         [100, 150, 200, 300],
    'model__max_leaf_nodes':   [15, 31, 63, 127],
    'model__min_samples_leaf': [10, 20, 50, 100],
    'model__l2_regularization': [0.0, 0.1, 0.5, 1.0],
    'model__max_depth':        [3, 5, 7, None],
}

print("RandomizedSearchCV — HistGradientBoostingClassifier")
print(f"n_iter=15 × cv={CV_FOLDS} = 75 fits (vs ~5000+ no GridSearch)")
print("Executando...\n")

search_hgbt = RandomizedSearchCV(
    modelos['HistGradientBoosting'],
    param_dist_hgbt,
    n_iter=15,
    cv=CV_FOLDS,
    scoring=SCORING,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=0,
    refit=True,
)
search_hgbt.fit(X_cv, y_cv)

hgbt_default_auc = cv_results['HistGradientBoosting']['auc_mean']
hgbt_tuned_auc   = search_hgbt.best_score_

print(f"Melhores hiperparâmetros : {search_hgbt.best_params_}")
print(f"AUC padrão  (cv=5)       : {hgbt_default_auc:.4f} ± {cv_results['HistGradientBoosting']['auc_std']:.4f}")
print(f"AUC ajustado (cv=5)      : {hgbt_tuned_auc:.4f}")
delta_hgbt = hgbt_tuned_auc - hgbt_default_auc
print(f"Variação                 : {delta_hgbt:+.4f} ({'melhora' if delta_hgbt > 0.001 else 'marginal' if delta_hgbt > 0 else 'sem ganho'})")

### 4.3 Comparação: Padrão vs. Ajustado

O ajuste de hiperparâmetros tipicamente traz **ganhos modestos** (0.001–0.010 em AUC) quando os defaults já são razoáveis — o que é comum nos algoritmos do sklearn. Ganhos maiores indicam que os parâmetros padrão não eram adequados para este dataset.

**Contexto:** Para o `LogisticRegression`, o parâmetro mais crítico é `C` (regularização): se o valor padrão (C=1) já for próximo do ótimo, o ganho será pequeno. Para o `HistGradientBoosting`, o `learning_rate` e `max_iter` interagem fortemente — aqui há mais espaço para ganhos.

In [ ]:
# Tabela comparativa: padrão vs. ajustado
comparacao = pd.DataFrame([
    {
        'Modelo':       'LogisticRegression',
        'AUC Padrão':   round(lr_default_auc,  4),
        'AUC Ajustado': round(lr_tuned_auc,    4),
        'Variação':     round(lr_tuned_auc - lr_default_auc, 4),
    },
    {
        'Modelo':       'HistGradientBoosting',
        'AUC Padrão':   round(hgbt_default_auc, 4),
        'AUC Ajustado': round(hgbt_tuned_auc,   4),
        'Variação':     round(hgbt_tuned_auc - hgbt_default_auc, 4),
    },
])
display(comparacao)

# Gráfico side-by-side
fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(2)
w = 0.35
b1 = ax.bar(x - w/2, comparacao['AUC Padrão'],  w, label='Padrão',   color='steelblue', alpha=0.85, edgecolor='black')
b2 = ax.bar(x + w/2, comparacao['AUC Ajustado'], w, label='Ajustado', color='coral',     alpha=0.85, edgecolor='black')
ax.set_xticks(x)
ax.set_xticklabels(comparacao['Modelo'])
ax.set_ylabel('AUC-ROC (CV=5, n=50k)')
ax.set_title('Impacto do Ajuste de Hiperparâmetros', fontsize=12, fontweight='bold')
ymin = min(comparacao['AUC Padrão'].min(), comparacao['AUC Ajustado'].min()) - 0.02
ymax = max(comparacao['AUC Padrão'].max(), comparacao['AUC Ajustado'].max()) + 0.02
ax.set_ylim(ymin, ymax)
ax.legend()
for b in list(b1) + list(b2):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.0005,
            f'{b.get_height():.4f}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

print(f"\nMelhor modelo ajustado:")
print(f"  LR ajustado  : AUC = {lr_tuned_auc:.4f}")
print(f"  HGBT ajustado: AUC = {hgbt_tuned_auc:.4f}")
print(f"  Vencedor     : {'HistGradientBoosting' if hgbt_tuned_auc >= lr_tuned_auc else 'LogisticRegression'}")

---
## SEÇÃO 5 — Escolha do Modelo Final

A escolha considera quatro dimensões:

| Critério | Peso | Descrição |
|---|---|---|
| **AUC-ROC (CV ajustado)** | Alto | Métrica principal do projeto — maior é melhor |
| **Estabilidade (desvio std)** | Alto | Modelos instáveis geram carteiras imprevisíveis |
| **Interpretabilidade** | Médio | Crédito tem exigências regulatórias (Basel III) — auditores precisam entender o modelo |
| **Complexidade computacional** | Baixo | Contexto de scoring em lote — latência não é crítica |

**Regra de decisão objetiva:**
- Se HistGBT superar LR em **≥ 0.005 AUC** → HistGBT (ganho prático relevante justifica complexidade)
- Caso contrário → LR (desempenho equivalente + interpretabilidade superior para crédito)

### 5.1 Modelo Final Escolhido e Hiperparâmetros

O código abaixo aplica a regra de decisão e exibe o modelo escolhido.

### 5.2 Justificativa Comparativa

**LogisticRegression (ajustado):**
- Prós: alta interpretabilidade (coeficientes = odds-ratio), extremamente estável, rápido
- Contras: assume linearidade — pode não capturar interações complexas entre features de risco

**HistGradientBoostingClassifier (ajustado):**
- Prós: melhor AUC em geral; captura relações não-lineares e interações; regularização L2 nas folhas
- Contras: caixa-preta — requer SHAP/LIME para explicabilidade; mais sensível a hiperparâmetros

**Por que 0.005 como threshold:** Em produção, diferenças de AUC < 0.005 são estatisticamente inseparáveis (especialmente com apenas 50k amostras no CV). A complexidade adicional do HistGBT (maior tempo de treino, menor interpretabilidade, mais hiperparâmetros) só se justifica quando a diferença de performance é concretamente perceptível.

In [ ]:
# Aplicar a regra de decisão para escolher o modelo final
print("=" * 60)
print("ESCOLHA DO MODELO FINAL")
print("=" * 60)
print(f"LR ajustado  : AUC = {lr_tuned_auc:.4f}")
print(f"HGBT ajustado: AUC = {hgbt_tuned_auc:.4f}")
print(f"Diferença    : {hgbt_tuned_auc - lr_tuned_auc:+.4f}")
print()

if hgbt_tuned_auc - lr_tuned_auc >= 0.005:
    melhor_nome     = 'HistGradientBoosting'
    melhor_pipeline = search_hgbt.best_estimator_
    melhor_params   = {k.replace('model__', ''): v for k, v in search_hgbt.best_params_.items()}
    auc_cv_melhor   = hgbt_tuned_auc
    razao = "AUC superior em >=0.005 — ganho prático relevante justifica a complexidade"
else:
    melhor_nome     = 'LogisticRegression'
    melhor_pipeline = search_lr.best_estimator_
    melhor_params   = {k.replace('model__', ''): v for k, v in search_lr.best_params_.items()}
    auc_cv_melhor   = lr_tuned_auc
    razao = "Diferença de AUC < 0.005 — interpretabilidade da LR é preferível no contexto de crédito"

print(f"Modelo escolhido : {melhor_nome}")
print(f"Razão            : {razao}")
print(f"\nHiperparâmetros finais:")
for k, v in sorted(melhor_params.items()):
    print(f"  {k}: {v}")
print(f"\nAUC estimada (CV): {auc_cv_melhor:.4f}")
print(f"\n{melhor_pipeline}")

---
## SEÇÃO 6 — Avaliação Final no Conjunto de Teste

> **⚠️ REGRA SAGRADA:** O conjunto de teste (`X_test_final`, `y_test`) é usado **uma única vez**, nesta seção. Qualquer ajuste realizado após ver esses resultados invalidaria a avaliação.

O modelo final é treinado no **conjunto de treino completo** (246k linhas — não na subamostra de 50k) e avaliado no conjunto de teste.

**Métricas reportadas:**
- AUC-ROC — comparar com estimativa do CV
- Acurácia, Precision, Recall, F1 por classe
- Matriz de Confusão — identificar FP e FN com contexto de negócio
- Curva ROC — poder discriminativo visual
- Classification Report — visão detalhada por classe

In [ ]:
# ⚠ CONJUNTO DE TESTE USADO UMA ÚNICA VEZ — NÃO EXECUTAR ESTA SEÇÃO MAIS DE UMA VEZ ⚠

# Treinar o modelo final no conjunto de treino COMPLETO (246k linhas)
print(f"Treinando {melhor_nome} no conjunto completo (X_train_final: {X_train_final.shape})...")
melhor_pipeline.fit(X_train_final, y_train)
print("Treinamento concluído.\n")

# Previsões no conjunto de teste — UMA ÚNICA VEZ
y_pred       = melhor_pipeline.predict(X_test_final)
y_pred_proba = melhor_pipeline.predict_proba(X_test_final)[:, 1]

print(f"Previsões geradas: {len(y_pred)} amostras")
print(f"Distribuição prevista: 0={( y_pred==0).sum():,} ({(y_pred==0).mean():.1%}) | "
      f"1={(y_pred==1).sum():,} ({(y_pred==1).mean():.1%})")
print(f"Real no teste:        0={( y_test==0).sum():,} ({(y_test==0).mean():.1%}) | "
      f"1={(y_test==1).sum():,} ({(y_test==1).mean():.1%})")

In [ ]:
# Métricas de avaliação final
auc_test  = roc_auc_score(y_test, y_pred_proba)
f1_test   = f1_score(y_test, y_pred, average='macro')
acc_test  = accuracy_score(y_test, y_pred)
prec_0    = precision_score(y_test, y_pred, pos_label=0)
rec_0     = recall_score(y_test, y_pred, pos_label=0)
prec_1    = precision_score(y_test, y_pred, pos_label=1)
rec_1     = recall_score(y_test, y_pred, pos_label=1)

metricas_df = pd.DataFrame({
    'Métrica': ['AUC-ROC', 'F1-Macro', 'Acurácia',
                'Precision (classe 0)', 'Recall (classe 0)',
                'Precision (classe 1)', 'Recall (classe 1)'],
    'Valor no Teste': [round(auc_test, 4), round(f1_test, 4), round(acc_test, 4),
                       round(prec_0, 4), round(rec_0, 4),
                       round(prec_1, 4), round(rec_1, 4)],
})
display(metricas_df)

print(f"\nComparação CV vs Teste:")
print(f"  AUC-ROC (CV estimado) : {auc_cv_melhor:.4f}")
print(f"  AUC-ROC (Teste real)  : {auc_test:.4f}")
diff = auc_test - auc_cv_melhor
print(f"  Diferença             : {diff:+.4f} ({'leve ganho' if diff > 0.005 else 'consistente' if abs(diff) <= 0.01 else 'leve queda'  if diff > -0.03 else 'queda relevante'})")

In [ ]:
# Matriz de Confusão
fig, ax = plt.subplots(figsize=(7, 5))
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['0 (Adimplente)', '1 (Inadimplente)']
)
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title(f'Matriz de Confusão — {melhor_nome}', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"Verdadeiro Negativo (TN): {tn:,}  — bons pagadores corretamente rejeitados (não, aprovados)")
print(f"Falso Positivo (FP)     : {fp:,}  — bons pagadores negados (custo de oportunidade)")
print(f"Falso Negativo (FN)     : {fn:,}  — inadimplentes aprovados (PERDA FINANCEIRA DIRETA)")
print(f"Verdadeiro Positivo (TP): {tp:,}  — inadimplentes corretamente detectados")
print(f"\nTaxa de detecção classe 1 (Recall): {tp/(tp+fn):.1%} ({tp} de {tp+fn} inadimplentes detectados)")
print(f"Precision classe 1               : {tp/(tp+fp):.1%} ({tp} de {tp+fp} predições positivas corretas)")

In [ ]:
# Curva ROC e AUC
fig, ax = plt.subplots(figsize=(8, 6))

RocCurveDisplay.from_predictions(
    y_test, y_pred_proba,
    name=f'{melhor_nome} (AUC = {auc_test:.4f})',
    ax=ax, color='steelblue'
)
ax.plot([0, 1], [0, 1], 'k--', linewidth=0.8, label='Classificador aleatório (AUC = 0.50)')
ax.fill_between([0, 1], [0, 0], [0, 1], alpha=0.05, color='red')
ax.set_title('Curva ROC — Avaliação Final no Conjunto de Teste', fontsize=12, fontweight='bold')
ax.set_xlabel('Taxa de Falso Positivo (FPR) = FP / (FP + TN)')
ax.set_ylabel('Taxa de Verdadeiro Positivo (TPR = Recall) = TP / (TP + FN)')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Interpretação da AUC = {auc_test:.4f}:")
print(f"  Em {auc_test:.1%} dos pares (inadimplente, adimplente) sorteados aleatoriamente,")
print(f"  o modelo atribui probabilidade de inadimplência maior ao inadimplente.")
print(f"  (AUC=0.50 → aleatório; AUC=1.00 → discriminação perfeita)")

In [ ]:
print("=" * 65)
print(f"CLASSIFICATION REPORT — {melhor_nome}")
print("=" * 65)
print(classification_report(
    y_test, y_pred,
    target_names=['0 (Adimplente)', '1 (Inadimplente)']
))

### 6.1 Análise: Teste vs. Cross-Validation

**Interpretação da diferença CV → Teste:**

| Diferença | Diagnóstico |
|---|---|
| < 0.01 | Normal — modelo generaliza bem |
| 0.01–0.03 | Atenção — leve overfitting ou variação de distribuição |
| > 0.03 | Problema — overfitting ou possível data leakage |

**Contexto desta sprint:** O CV foi feito em subamostra de 50k (não nos 246k completos), o que pode subestimar ou superestimar ligeiramente a AUC esperada no treino completo. O modelo final foi treinado nos 246k, o que geralmente melhora a estimativa.

**Análise dos erros (Matriz de Confusão):**
- **Falsos Negativos (FN):** inadimplentes não detectados → risco financeiro direto para a instituição
- **Falsos Positivos (FP):** bons pagadores recusados → custo de oportunidade (não receber os juros)

O `class_weight='balanced'` aumenta o Recall da classe 1 em relação a modelos sem ponderação — o modelo "sacrifica" parte da precisão para detectar mais inadimplentes reais. Esse trade-off é adequado para o contexto de risco de crédito, onde FN é muito mais custoso que FP.

**Se a diferença CV → Teste for > 0.02:** uma causa provável é que a feature selection (RF no X_train completo, fora do CV) introduziu um leve data leakage informativo. Para eliminar completamente esse viés, a seleção deveria ser embutida dentro do CV — objetivo da Sprint 4.

---
## SEÇÃO 7 — Persistência do Modelo

Salvamos o pipeline final com `joblib` para que possa ser reutilizado em produção ou na Sprint 4 sem necessidade de re-treinamento.

**Por que `joblib` e não `pickle`:** `joblib` é mais eficiente para objetos NumPy grandes (arrays de coeficientes do modelo), e é o formato recomendado pelo sklearn para persistência de modelos em produção.

In [ ]:
model_path = 'modelo_projeto.pkl'
joblib.dump(melhor_pipeline, model_path)

model_size = os.path.getsize(model_path) / (1024**2)
print(f"Modelo salvo em '{model_path}' ({model_size:.2f} MB)")
print(f"   Tipo de modelo : {melhor_pipeline.named_steps['model'].__class__.__name__}")
print(f"   Hiperparâmetros: {melhor_params}")
print(f"   AUC estimada   : {auc_cv_melhor:.4f} (CV) | {auc_test:.4f} (Teste)")

In [ ]:
# Carregar e verificar que as previsões são bit-a-bit idênticas
modelo_carregado  = joblib.load(model_path)

y_pred_reloaded   = modelo_carregado.predict(X_test_final)
y_proba_reloaded  = modelo_carregado.predict_proba(X_test_final)[:, 1]

pred_match  = (y_pred == y_pred_reloaded).all()
proba_match = np.allclose(y_pred_proba, y_proba_reloaded)

print(f"Previsões binárias idênticas : {pred_match}")
print(f"Probabilidades idênticas     : {proba_match}")
print(f"AUC original  : {roc_auc_score(y_test, y_pred_proba):.6f}")
print(f"AUC reloaded  : {roc_auc_score(y_test, y_proba_reloaded):.6f}")

assert pred_match and proba_match, "ERRO: previsões do modelo carregado diferem do original!"
print(f"\nModelo carregado de '{model_path}' produz previsões idênticas.")
print("   Pipeline pronto para uso na Sprint 4.")

---
## Conclusão da Sprint 3

### Resumo dos Resultados

| Etapa | Resultado |
|---|---|
| **Baseline** | AUC ≈ 0.50 — equivalente ao aleatório |
| **Comparação de 4 modelos** | LR e HistGBT como melhores (AUC > 0.70); DT com alta variância |
| **Ajuste LR** (GridSearchCV) | Ganho incremental via otimização de C e solver |
| **Ajuste HistGBT** (RandomizedSearchCV) | Ganho incremental via learning_rate, max_iter e regularização |
| **Modelo final** | Escolhido por AUC + estabilidade + interpretabilidade |
| **Avaliação no teste** | AUC consistente com estimativa do CV |
| **Persistência** | Pipeline salvo em `modelo_projeto.pkl` e verificado |

### Principais Decisões Documentadas

1. **Métrica AUC-ROC** — robusta ao desbalanceamento (8% classe 1); não depende do threshold
2. **`class_weight='balanced'`** — corrige viés de classes sem undersamplear dados
3. **Subamostra 50k para CV** — equilíbrio entre rigor estatístico e tempo de execução razoável
4. **Desvio padrão como critério** — um modelo instável é operacionalmente arriscado mesmo com boa média
5. **Threshold 0.005 para escolha LR vs HistGBT** — diferenças menores não justificam perda de interpretabilidade

### Próximos Passos — Sprint 4

- **Análise de erros profunda:** perfil dos FN (inadimplentes não detectados) — que características eles têm?
- **Interpretabilidade:** SHAP values para explicar previsões individuais e identificar features mais influentes
- **Otimização de threshold:** ajustar o ponto de corte além de 0.5 para maximizar Recall (reduzir FN)
- **Incorporação de tabelas auxiliares:** `bureau.csv`, `previous_application.csv` (mais features preditivas)
- **Conclusões e apresentação final do projeto**